# Linear and Integer Optimization - Theory and Practice

In [1]:
import pandas as pd
import gurobipy as gb
from gurobipy import GRB

In [2]:
products = ["Long", "Short"]
materials = ["Wood", "Boxes"]
profit = pd.Series([3000, 2000], index=products, name="profit")
max_production = 9
wood_requirement = pd.Series([3, 1], index=products, name="wood_requirement")
wood_max = 18
box_availability = pd.Series([7, 6], index=products, name="box_availability")

In [3]:
m1 = gb.Model('Matchbox')

Restricted license - for non-production use only - expires 2026-11-23


In [4]:
# Decision Variables
x = {}
for p in products:
    x[p] = m1.addVar(name="quantity_" + p, vtype=GRB.INTEGER)

In [5]:
m1.update()

In [6]:
max_prod = m1.addConstr((gb.quicksum(x[p] for p in products) <= max_production),
                         name="max_prod")

In [7]:
wood_capacity = m1.addConstr((gb.quicksum(x[p] * wood_requirement[p] for p in products) <= wood_max),
                             name="wood_capacity")

In [8]:
box_capacity = m1.addConstrs((x[p] <= box_availability[p] for p in products), name="box_capacity")

In [9]:
# Objective
m1.setObjective(gb.quicksum(x[p] * profit[p] for p in products), GRB.MAXIMIZE)
m1.update()

In [10]:
m1.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i5-13450HX, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 4 rows, 2 columns and 6 nonzeros
Model fingerprint: 0x2aebe7a7
Variable types: 0 continuous, 2 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [2e+03, 3e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e+00, 2e+01]
Found heuristic solution: objective 18000.000000
Presolve removed 2 rows and 0 columns
Presolve time: 0.01s
Presolved: 2 rows, 2 columns, 4 nonzeros
Variable types: 0 continuous, 2 integer (0 binary)

Root relaxation: objective 2.250000e+04, 2 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 22500.0

In [11]:
x_values = pd.Series(m1.getAttr('X', x), name="quantity", index=products)

In [12]:
x_values

Long     4.0
Short    5.0
Name: quantity, dtype: float64

In [13]:
for v in m1.getVars():
    print(f"{v.VarName} = {v.X}")

quantity_Long = 4.0
quantity_Short = 5.0


In [18]:
print(f"Profit: {m1.ObjVal}")

Profit: 22000.0
